# CV Lab 09 — Object Detection & Pose Estimation

**One notebook for the manual and Lab Task PDF (AI-4002).**  
Designed for **Google Colab**. Run cells from top to bottom.

## What this notebook covers
1. YOLO live/webcam-style detection and image/video detection.
2. YOLOv3, SSD MobileNet, and RetinaNet on the same images.
3. Bounding boxes, labels, confidence scores, FPS, and inference time.
4. A simple comparison of speed, confidence, model size, and CPU/GPU use.
5. Car detection on street images.
6. A safety-monitor example for person, bike/motorcycle, and helmet.
7. Human pose estimation with a skeleton overlay.

**Important notes**
- The pretrained COCO models can detect common objects, but they do **not** reliably detect helmets. Helmet compliance needs a helmet-trained model. This notebook includes the logic and a clear place to load custom weights.
- YOLOv3 weights, SSD weights, RetinaNet weights, and MediaPipe may need internet access the first time.
- In Colab, webcam access is handled through a browser snapshot helper. If you use a local Jupyter runtime instead, use the optional local webcam cell.

## 1. Install packages

In [ ]:
!pip -q install ultralytics opencv-python-headless tensorflow tensorflow-hub torch torchvision mediapipe pandas matplotlib

## 2. Imports and device check

In [ ]:
import os, time, math, urllib.request, zipfile, pathlib, warnings
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU (CPU mode)")

## 3. Add your test images or video

Upload **one or more common images** (street/lab images work well) and optionally a video.  
Use the same image files for all three detectors so the comparison is fair.

In [ ]:
from google.colab import files
uploaded = files.upload()

DATA_DIR = "/content/cv_lab09_data"
os.makedirs(DATA_DIR, exist_ok=True)
for name, data in uploaded.items():
    path = os.path.join(DATA_DIR, name)
    with open(path, "wb") as f:
        f.write(data)

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")
image_paths = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.lower().endswith(IMAGE_EXTS)]
video_paths = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.lower().endswith(VIDEO_EXTS)]
print("Images:", image_paths)
print("Videos:", video_paths)
if not image_paths:
    print("No images uploaded yet. Run this cell again after uploading at least one image.")

## 4. Helper functions

In [ ]:
def show_bgr(img, title=None, figsize=(12, 7)):
    if img is None:
        print("Image could not be loaded.")
        return
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if title: plt.title(title)
    plt.axis("off")
    plt.show()

def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024) if path and os.path.exists(path) else np.nan

def safe_read_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    return img

if image_paths:
    sample_img = safe_read_image(image_paths[0])
    show_bgr(sample_img, "Sample input image")

## 5. YOLOv8 detection (Ultralytics)

The lab manual demonstrates Ultralytics YOLO and drawing boxes, labels, and confidence.  
We use a small pretrained model (`yolov8n.pt`) so it is practical in Colab.

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8n.pt")  # downloads pretrained weights on first run
print("YOLO loaded.")

In [ ]:
def run_yolo_image(model, image_path, conf=0.25, show=True):
    img = safe_read_image(image_path)
    start = time.perf_counter()
    result = model.predict(source=img, conf=conf, verbose=False)[0]
    elapsed = time.perf_counter() - start
    annotated = result.plot()
    detections = []
    names = model.names
    if result.boxes is not None:
        for b in result.boxes:
            cls_id = int(b.cls.item())
            score = float(b.conf.item())
            x1, y1, x2, y2 = map(float, b.xyxy[0].tolist())
            detections.append({
                "class": names[cls_id], "confidence": score,
                "x1": x1, "y1": y1, "x2": x2, "y2": y2
            })
    if show:
        show_bgr(annotated, f"YOLO — {os.path.basename(image_path)}")
    return detections, elapsed, annotated

yolo_results = []
for p in image_paths:
    dets, seconds, annotated = run_yolo_image(yolo_model, p)
    yolo_results.append({"image": os.path.basename(p), "model": "YOLOv8n",
                         "inference_seconds": seconds, "fps": 1/seconds if seconds else np.nan,
                         "detections": len(dets),
                         "mean_confidence": np.mean([d["confidence"] for d in dets]) if dets else np.nan})
pd.DataFrame(yolo_results)

## 6. YOLOv3 detection

Ultralytics provides a pretrained YOLOv3 model. We use `yolov3u.pt` and the same images.
If the weight download is unavailable in your environment, skip this section and record that limitation.

In [ ]:
try:
    yolov3_model = YOLO("yolov3u.pt")
    print("YOLOv3 loaded.")
except Exception as e:
    yolov3_model = None
    print("YOLOv3 could not be loaded:", repr(e))

In [ ]:
yolov3_results = []
if yolov3_model is not None:
    for p in image_paths:
        try:
            dets, seconds, annotated = run_yolo_image(yolov3_model, p)
            yolov3_results.append({"image": os.path.basename(p), "model": "YOLOv3u",
                                   "inference_seconds": seconds, "fps": 1/seconds if seconds else np.nan,
                                   "detections": len(dets),
                                   "mean_confidence": np.mean([d["confidence"] for d in dets]) if dets else np.nan})
        except Exception as e:
            print("YOLOv3 error for", p, ":", e)
pd.DataFrame(yolov3_results)

## 7. SSD MobileNet V2 FPNLite

This follows the manual's TensorFlow Hub SSD approach. It loads the public SavedModel and runs it on the same images.

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

SSD_HANDLE = "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/fpnlite_320x320/1"
try:
    ssd_detector = hub.load(SSD_HANDLE)
    print("SSD model loaded.")
except Exception as e:
    ssd_detector = None
    print("SSD model could not be loaded:", repr(e))

In [ ]:
# COCO label names used by common TensorFlow Object Detection models.
COCO_LABELS = {
    1:"person", 2:"bicycle", 3:"car", 4:"motorcycle", 5:"airplane", 6:"bus",
    7:"train", 8:"truck", 9:"boat", 10:"traffic light", 13:"stop sign",
    17:"cat", 18:"dog", 44:"bottle", 62:"chair", 63:"couch", 64:"potted plant",
    67:"dining table", 72:"tv", 73:"laptop", 74:"mouse", 77:"cell phone"
}

def run_ssd_image(detector, image_path, threshold=0.25, show=True):
    bgr = safe_read_image(image_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    tensor = tf.convert_to_tensor(rgb, dtype=tf.uint8)[tf.newaxis, ...]
    start = time.perf_counter()
    output = detector(tensor)
    elapsed = time.perf_counter() - start
    boxes = output["detection_boxes"][0].numpy()
    scores = output["detection_scores"][0].numpy()
    classes = output["detection_classes"][0].numpy().astype(int)
    h, w = bgr.shape[:2]
    annotated = bgr.copy()
    dets = []
    for box, score, cls_id in zip(boxes, scores, classes):
        if score < threshold: continue
        ymin, xmin, ymax, xmax = box
        x1, y1, x2, y2 = int(xmin*w), int(ymin*h), int(xmax*w), int(ymax*h)
        label = COCO_LABELS.get(cls_id, f"class_{cls_id}")
        cv2.rectangle(annotated, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(annotated, f"{label} {score:.2f}", (x1,max(20,y1-7)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        dets.append({"class":label, "confidence":float(score),
                     "x1":x1,"y1":y1,"x2":x2,"y2":y2})
    if show: show_bgr(annotated, f"SSD MobileNet — {os.path.basename(image_path)}")
    return dets, elapsed, annotated

ssd_results = []
if ssd_detector is not None:
    for p in image_paths:
        try:
            dets, seconds, annotated = run_ssd_image(ssd_detector, p)
            ssd_results.append({"image":os.path.basename(p), "model":"SSD MobileNet",
                                "inference_seconds":seconds, "fps":1/seconds if seconds else np.nan,
                                "detections":len(dets),
                                "mean_confidence":np.mean([d["confidence"] for d in dets]) if dets else np.nan})
        except Exception as e:
            print("SSD error for", p, ":", e)
pd.DataFrame(ssd_results)

## 8. RetinaNet ResNet-50 FPN

This uses the pretrained torchvision RetinaNet model and the same images.

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn, RetinaNet_ResNet50_FPN_Weights

try:
    retina_weights = RetinaNet_ResNet50_FPN_Weights.DEFAULT
    retina_model = retinanet_resnet50_fpn(weights=retina_weights).to(DEVICE).eval()
    retina_transform = retina_weights.transforms()
    retina_categories = retina_weights.meta["categories"]
    print("RetinaNet loaded.")
except Exception as e:
    retina_model = None
    print("RetinaNet could not be loaded:", repr(e))

In [ ]:
def run_retinanet_image(model, image_path, threshold=0.5, show=True):
    bgr = safe_read_image(image_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    x = retina_transform(pil).to(DEVICE)
    start = time.perf_counter()
    with torch.no_grad():
        pred = model([x])[0]
    elapsed = time.perf_counter() - start
    annotated = bgr.copy()
    dets = []
    for box, score, label in zip(pred["boxes"].cpu().numpy(),
                                 pred["scores"].cpu().numpy(),
                                 pred["labels"].cpu().numpy()):
        if score < threshold: continue
        x1,y1,x2,y2 = map(int, box)
        name = retina_categories[int(label)]
        cv2.rectangle(annotated,(x1,y1),(x2,y2),(255,100,0),2)
        cv2.putText(annotated,f"{name} {score:.2f}",(x1,max(20,y1-7)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,100,0),2)
        dets.append({"class":name,"confidence":float(score),
                     "x1":x1,"y1":y1,"x2":x2,"y2":y2})
    if show: show_bgr(annotated, f"RetinaNet — {os.path.basename(image_path)}")
    return dets, elapsed, annotated

retina_results = []
if retina_model is not None:
    for p in image_paths:
        try:
            dets, seconds, annotated = run_retinanet_image(retina_model, p)
            retina_results.append({"image":os.path.basename(p),"model":"RetinaNet R50-FPN",
                                   "inference_seconds":seconds,"fps":1/seconds if seconds else np.nan,
                                   "detections":len(dets),
                                   "mean_confidence":np.mean([d["confidence"] for d in dets]) if dets else np.nan})
        except Exception as e:
            print("RetinaNet error for",p,":",e)
pd.DataFrame(retina_results)

## 9. Compare all detectors

**FPS** here is calculated as `1 / inference time` for one image. It does not include file loading or display time.
For a stronger benchmark, run several images and compare the average values.

In [ ]:
all_results = yolo_results + yolov3_results + ssd_results + retina_results
comparison = pd.DataFrame(all_results)
if len(comparison):
    display(comparison)
    summary = comparison.groupby("model").agg(
        avg_inference_seconds=("inference_seconds","mean"),
        avg_fps=("fps","mean"),
        avg_detections=("detections","mean"),
        avg_confidence=("mean_confidence","mean")
    ).reset_index()
    display(summary)
else:
    print("No benchmark results yet. Check that images were uploaded and model cells ran.")

### Model size (weight files)

This reports local weight file sizes where the library exposes a known file path. Some libraries cache weights in different places, so a missing value does not mean the model has no size.

In [ ]:
def find_weight_file(model_obj):
    # Ultralytics model weights
    try:
        p = getattr(model_obj, "ckpt_path", None)
        if p and os.path.exists(p): return p
    except: pass
    return None

size_rows = []
for name, obj in [("YOLOv8n", yolo_model),
                  ("YOLOv3u", yolov3_model),
                  ("SSD MobileNet", None),
                  ("RetinaNet R50-FPN", None)]:
    path = find_weight_file(obj) if obj is not None else None
    size_rows.append({"model":name,"weight_file":path,
                      "size_MB":file_size_mb(path) if path else np.nan})
pd.DataFrame(size_rows)

## 10. Task 1 — Car detection on a street dataset

Upload street images in Section 3. This cell filters YOLO detections to cars, buses, and trucks.
For a full custom street dataset, upload its images and repeat the evaluation over all files.

In [ ]:
VEHICLE_CLASSES = {"car", "bus", "truck", "motorcycle", "bicycle"}

def detect_vehicles_yolo(image_path, model=yolo_model, conf=0.25):
    dets, seconds, annotated = run_yolo_image(model, image_path, conf=conf, show=False)
    vehicles = [d for d in dets if d["class"].lower() in VEHICLE_CLASSES]
    out = safe_read_image(image_path)
    for d in vehicles:
        x1,y1,x2,y2 = map(int,[d["x1"],d["y1"],d["x2"],d["y2"]])
        cv2.rectangle(out,(x1,y1),(x2,y2),(0,255,255),2)
        cv2.putText(out,f'{d["class"]} {d["confidence"]:.2f}',(x1,max(20,y1-7)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,255),2)
    show_bgr(out, "Street vehicle detection")
    return vehicles

for p in image_paths:
    print(os.path.basename(p), detect_vehicles_yolo(p))

## 11. Task 2 — Multi-class safety monitor

The standard COCO model can detect **person, bicycle, and motorcycle**. It does not have a helmet class.
The cell below shows the compliance logic. For real helmet checks, train or obtain a custom YOLO model with classes such as `helmet` and `no_helmet`, then set `CUSTOM_SAFETY_WEIGHTS` to its `.pt` file.

In [ ]:
# Optional: upload your own helmet-trained YOLO weights, then set this path.
CUSTOM_SAFETY_WEIGHTS = None  # Example: "/content/best.pt"

safety_model = YOLO(CUSTOM_SAFETY_WEIGHTS) if CUSTOM_SAFETY_WEIGHTS else yolo_model

def safety_monitor(image_path, model=safety_model, conf=0.25):
    dets, seconds, _ = run_yolo_image(model, image_path, conf=conf, show=False)
    counts = {}
    for d in dets:
        label = d["class"].lower()
        counts[label] = counts.get(label, 0) + 1

    people = counts.get("person", 0)
    bikes = counts.get("bicycle", 0) + counts.get("motorcycle", 0)
    helmets = counts.get("helmet", 0)
    no_helmets = counts.get("no_helmet", 0) + counts.get("without helmet", 0)

    print("People detected:", people)
    print("Bikes / motorcycles detected:", bikes)
    if CUSTOM_SAFETY_WEIGHTS:
        print("Helmets detected:", helmets)
        print("No-helmet detections:", no_helmets)
        print("Basic monitor status:", "POSSIBLE NON-COMPLIANCE" if no_helmets else "REVIEW REQUIRED")
    else:
        print("Helmet status: UNKNOWN — current pretrained model has no helmet class.")
        print("Compliance cannot be decided from person/bike detections alone.")
    print("Inference time:", round(seconds, 4), "seconds")

for p in image_paths:
    print("\n", os.path.basename(p))
    safety_monitor(p)

## 12. Live webcam detection (Google Colab)

Colab does not directly connect to `cv2.VideoCapture(0)`. This browser helper takes a webcam snapshot and sends it to Python.
Run the cell, allow camera access, then run YOLO on the captured frame. Repeat to capture another frame.

In [ ]:
from google.colab.output import eval_js
from IPython.display import Javascript, display as ipy_display
from google.colab.patches import cv2_imshow
from base64 import b64decode

def take_webcam_snapshot():
    js = Javascript('''
    async function takePhoto() {
      const div = document.createElement('div');
      const video = document.createElement('video');
      const button = document.createElement('button');
      button.textContent = 'Capture frame';
      div.appendChild(video); div.appendChild(button);
      document.body.appendChild(div);
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream;
      await video.play();
      await new Promise(resolve => button.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth; canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getTracks().forEach(track => track.stop());
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.9);
    }
    takePhoto();
    ''')
    data = eval_js(js)
    binary = b64decode(data.split(",")[1])
    arr = np.frombuffer(binary, dtype=np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

webcam_frame = take_webcam_snapshot()
webcam_result = yolo_model.predict(source=webcam_frame, conf=0.25, verbose=False)[0]
cv2_imshow(webcam_result.plot())
print("Detected objects:", len(webcam_result.boxes))

## 13. Optional local webcam loop (run only in local Jupyter, not Colab)

This is the continuous webcam loop shown in the manual. Press **q** in the webcam window to stop.

In [ ]:
# Uncomment and run only on a computer with a webcam and a local GUI:
#
# cap = cv2.VideoCapture(0)
# while cap.isOpened():
#     ok, frame = cap.read()
#     if not ok: break
#     result = yolo_model.predict(frame, conf=0.25, verbose=False)[0]
#     cv2.imshow("YOLO Live", result.plot())
#     if cv2.waitKey(1) & 0xFF == ord("q"):
#         break
# cap.release()
# cv2.destroyAllWindows()

## 14. Real-time pose estimation (MediaPipe)

This follows the manual: read a frame, estimate human pose, draw the skeleton, and show landmark coordinates.

In [ ]:
import mediapipe as mp
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

def estimate_pose_image(bgr_image, show=True):
    rgb = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    with mp_pose.Pose(static_image_mode=True, model_complexity=1,
                      min_detection_confidence=0.5) as pose:
        results = pose.process(rgb)
    output = bgr_image.copy()
    coords = []
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(output, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        h,w = output.shape[:2]
        for idx, lm in enumerate(results.pose_landmarks.landmark):
            coords.append({"landmark_id":idx, "x_pixels":int(lm.x*w),
                           "y_pixels":int(lm.y*h), "visibility":float(lm.visibility)})
    if show: show_bgr(output, "MediaPipe pose estimation")
    return output, pd.DataFrame(coords)

if image_paths:
    pose_image, pose_coordinates = estimate_pose_image(safe_read_image(image_paths[0]))
    display(pose_coordinates.head(10))

## 15. Pose estimation from webcam snapshot (Colab)

Capture a person standing in view of the webcam. The output will show the skeleton and landmark coordinates.

In [ ]:
pose_webcam_frame = take_webcam_snapshot()
pose_output, pose_coords = estimate_pose_image(pose_webcam_frame)
display(pose_coords)

## 16. Final results and discussion

Use the measured table from Section 9. Your discussion should be based on your actual results, not a fixed claim.

Suggested points to discuss:
- **Real-time detection:** the model with the highest measured FPS is the fastest on your current hardware.
- **Small objects:** compare whether each model finds the same small objects. Confidence alone does not prove accuracy.
- **CPU-only systems:** compare CPU inference time and memory use. Smaller models are usually easier to run, but verify with your results.
- **Accuracy:** without ground-truth labels, this notebook reports confidence and detection counts, **not true accuracy**. To calculate precision/recall/mAP, you need labelled bounding boxes.
- **Model size:** report the actual downloaded weight-file size if available.

### Short conclusion template
Fill in the blanks after running the notebook:

> On our test images, ______ had the highest average FPS. ______ produced the highest average confidence, while ______ detected the most objects. For a real-time CPU-only system, ______ was the most suitable based on our measured speed and model size. These results are limited by the number of test images and the lack of ground-truth labels.

## 17. Save benchmark results

This saves the comparison table as a CSV file that you can download and use in your lab report.

In [ ]:
if len(comparison):
    comparison.to_csv("/content/cv_lab09_benchmark.csv", index=False)
    from google.colab import files
    files.download("/content/cv_lab09_benchmark.csv")
else:
    print("Run the model benchmark sections first.")